# XGBoost — Store Sales Forecasting

XGBoost არის gradient boosting: თანმიმდევრობით აშენებს პატარა გადაწყვეტილების ხეებს
და თითო ახალი ხე წინა ხეების შეცდომას ასწორებს. tabular მონაცემებზე ერთ-ერთი
ყველაზე ძლიერი მოდელია, ამიტომ დავიწყე მისით.

ამ notebook-ში:
- ვტვირთავ მონაცემებს და ვჰყოფ დროზე (ბოლო 12 კვირა = ვალიდაცია);
- ვუშვებ რამდენიმე ვარიანტს (სხვადასხვა ფიჩერი + ჰიპერპარამეტრი);
- ვადარებ WMAE-ით და საუკეთესოს ვინახავ wandb-ზე Pipeline-ად.

### 1. ბიბლიოთეკები

In [ ]:
import sys
import os
import warnings

sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
os.environ.setdefault("WANDB_SILENT", "true")

from xgboost import XGBRegressor

from src.data import load_raw
from src.metrics import wmae, holiday_weights
from src.pipeline import build_pipeline, RAW_COLS
from src.validation import time_holdout_split, cross_validate_wmae
from src.wandb_utils import init_run, log_pipeline

### 2. მონაცემების ჩატვირთვა

In [ ]:
raw = load_raw("data")

train = raw.train
features = raw.features
stores = raw.stores

print("train:", train.shape)
print("features:", features.shape)
print("stores:", stores.shape)

### 3. ვალიდაცია დროზე

random split აქ არ გამოდგება — ის მომავალ კვირებზე დაატრენინგებდა და წარსულს
იწინასწარმეტყველებდა (leakage). ამიტომ ბოლო 12 კვირას ვტოვებ ვალიდაციისთვის.

In [ ]:
tr_df, val_df = time_holdout_split(train, n_val_weeks=12)

print("train ნაწილი:", tr_df.shape)
print("validation ნაწილი:", val_df.shape)

### 4. Holiday წონები

მეტრიკა (WMAE) სადღესასწაულო კვირებს 5-ჯერ მეტ წონას აძლევს. იმავეს ვეუბნები
მოდელს `sample_weight`-ით, რომ იმ კვირების შეცდომას მეტი მნიშვნელობა მისცეს.

In [ ]:
w_tr = holiday_weights(tr_df["IsHoliday"])

print("holiday კვირების წილი:", round((w_tr == 5).mean(), 3))

### 5. ვარიანტები

თითო ვარიანტი ცვლის ან ფიჩერებს (`fe`), ან ჰიპერპარამეტრებს (`params`), ან ორივეს.
ასე ვნახავ რა უფრო მოქმედებს შედეგზე.

In [ ]:
COMMON = dict(tree_method="hist", random_state=42, n_jobs=-1)

EXPERIMENTS = [
    {
        "name": "XGBoost_v1_baseline",
        "fe": {},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 400,
                   "learning_rate": 0.05, "max_depth": 8,
                   "subsample": 0.8, "colsample_bytree": 0.8},
    },
    {
        "name": "XGBoost_v2_no_markdowns_deep",
        "fe": {"use_markdowns": False},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 400,
                   "learning_rate": 0.03, "max_depth": 10, "min_child_weight": 5,
                   "subsample": 0.8, "colsample_bytree": 0.8},
    },
    {
        "name": "XGBoost_v3_interactions_no_cyclical",
        "fe": {"add_interactions": True, "use_cyclical": False},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 400,
                   "learning_rate": 0.05, "max_depth": 8,
                   "subsample": 0.8, "colsample_bytree": 0.7},
    },
    {
        "name": "XGBoost_v4_rich_regularised",
        "fe": {"add_interactions": True},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 400,
                   "learning_rate": 0.03, "max_depth": 9, "reg_lambda": 2.0,
                   "subsample": 0.9, "colsample_bytree": 0.8},
    },
    {
        "name": "XGBoost_v5_no_markdowns_tuned",
        "fe": {"use_markdowns": False},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 700,
                   "learning_rate": 0.02, "max_depth": 10, "min_child_weight": 3,
                   "reg_lambda": 1.5, "subsample": 0.85, "colsample_bytree": 0.85},
    },
    {
        "name": "XGBoost_v6_shallow_many_trees",
        "fe": {"use_markdowns": False},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 1200,
                   "learning_rate": 0.02, "max_depth": 6, "min_child_weight": 2,
                   "subsample": 0.8, "colsample_bytree": 0.8},
    },
    {
        "name": "XGBoost_v7_squared_loss",
        "fe": {"use_markdowns": False},
        "params": {"objective": "reg:squarederror", "n_estimators": 700,
                   "learning_rate": 0.02, "max_depth": 10, "min_child_weight": 3,
                   "subsample": 0.85, "colsample_bytree": 0.85},
    },
    {
        "name": "XGBoost_v8_no_holiday_flags",
        "fe": {"use_markdowns": False, "use_holiday_flags": False},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 700,
                   "learning_rate": 0.02, "max_depth": 10, "min_child_weight": 3,
                   "reg_lambda": 1.5, "subsample": 0.85, "colsample_bytree": 0.85},
    },
    {
        "name": "XGBoost_v9_deep_strong_reg",
        "fe": {"use_markdowns": False},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 900,
                   "learning_rate": 0.02, "max_depth": 12, "min_child_weight": 4,
                   "reg_lambda": 3.0, "reg_alpha": 1.0,
                   "subsample": 0.8, "colsample_bytree": 0.8},
    },
    {
        "name": "XGBoost_v10_all_features_tuned",
        "fe": {"add_interactions": True},
        "params": {"objective": "reg:absoluteerror", "n_estimators": 800,
                   "learning_rate": 0.02, "max_depth": 10, "min_child_weight": 3,
                   "reg_lambda": 1.5, "subsample": 0.85, "colsample_bytree": 0.85},
    },
]

print("სულ ვარიანტი:", len(EXPERIMENTS))

### 6. თითო ვარიანტის გაშვება

თითო ვარიანტზე:
1. ვაწყობ Pipeline-ს (preprocessing + XGBoost);
2. ვატრენინგებ train ნაწილზე holiday წონებით;
3. ვამოწმებ validation-ზე WMAE-ით;
4. ვინახავ wandb-ზე.

In [ ]:
results = []

for exp in EXPERIMENTS:
    name = exp["name"]
    fe = exp["fe"]
    params = exp["params"]

    # ერთი config = ერთი wandb run
    config = {}
    config.update(fe)
    config.update(params)
    run = init_run(group="XGBoost_Training", job_type="experiment", name=name, config=config)

    # Pipeline = preprocessing + model
    model = XGBRegressor(**COMMON, **params)
    pipe = build_pipeline(model, features, stores, **fe)

    # ტრენინგი holiday წონებით
    pipe.fit(tr_df[RAW_COLS], tr_df["Weekly_Sales"], model__sample_weight=w_tr)

    # შეფასება validation-ზე
    pred = pipe.predict(val_df[RAW_COLS])
    score = wmae(val_df["Weekly_Sales"], pred, val_df["IsHoliday"])
    n_features = len(pipe.named_steps["finalize"].columns_)

    # ლოგირება
    run.summary["holdout_wmae"] = score
    run.summary["wmae_val"] = score
    run.summary["n_features"] = n_features
    log_pipeline(run, pipe, name="walmart_xgboost_" + name.split("_", 1)[1],
                 metadata={"holdout_wmae": score})
    run.finish()

    results.append((name, score, n_features))
    print(name, "->", round(score, 2), "WMAE |", n_features, "features")

### 7. საუკეთესო ვარიანტის მოძებნა

In [ ]:
best_name = None
best_score = float("inf")

for name, score, n_features in results:
    if score < best_score:
        best_score = score
        best_name = name

print("საუკეთესო:", best_name, "->", round(best_score, 2), "WMAE")

# ვიპოვო შესაბამისი config
best_exp = None
for exp in EXPERIMENTS:
    if exp["name"] == best_name:
        best_exp = exp
        break

### 8. Final — საუკეთესო კონფიგი მთელ train-ზე + CV + რეგისტრაცია

ახლა საუკეთესო კონფიგს ვამოწმებ walk-forward CV-ით და ვატრენინგებ **მთელ** train-ზე,
შემდეგ ვინახავ `walmart_xgboost:best`-ად.

In [ ]:
config = {}
config.update(best_exp["fe"])
config.update(best_exp["params"])

run = init_run(group="XGBoost_Training", job_type="final", name="XGBoost_Final", config=config)

# walk-forward CV
cv_pipe = build_pipeline(XGBRegressor(**COMMON, **best_exp["params"]),
                         features, stores, **best_exp["fe"])
cv = cross_validate_wmae(cv_pipe, train, n_splits=3, n_val_weeks=8)
print("CV mean WMAE:", round(cv["mean_wmae"], 2), "+/-", round(cv["std_wmae"], 2))

In [ ]:
# refit მთელ train-ზე
final_model = XGBRegressor(**COMMON, **best_exp["params"])
final_pipe = build_pipeline(final_model, features, stores, **best_exp["fe"])

w_all = holiday_weights(train["IsHoliday"])
final_pipe.fit(train[RAW_COLS], train["Weekly_Sales"], model__sample_weight=w_all)

run.summary["holdout_wmae"] = best_score
run.summary["cv_mean_wmae"] = cv["mean_wmae"]
run.summary["cv_std_wmae"] = cv["std_wmae"]

log_pipeline(run, final_pipe, name="walmart_xgboost",
             metadata={"holdout_wmae": best_score, "cv_mean_wmae": cv["mean_wmae"]},
             aliases=["best"])
run.finish()
print("დარეგისტრირდა: walmart_xgboost:best")

### შედეგები

| ვარიანტი | ფიჩერები | WMAE |
|---|---|---|
| v1 baseline | 27 | 2612 |
| v2 no-markdowns | 22 | 2294 |
| v3 interactions, −cyclical | 26 | 2830 |
| v4 rich + reg | 30 | 2620 |
| v5 tuned | 22 | 2158 |
| v6 shallow × many | 22 | 3340 |
| v7 squared loss | 22 | 1869 |
| v8 no holiday flags | 18 | 2153 |
| **v9 deep + strong reg** | 22 | **1869** |
| v10 all features | 30 | 2178 |

**რას ვხედავ:**
- markdown-ების მოშორება ეხმარება (2612 → 2294) — ბევრი NaN-ია და ხმაურს ამატებს.
- ყველაზე დიდი გაუმჯობესება მოიტანა ღრმა ხეებმა + ძლიერმა regularization-მა (v9).
- squared და absolute loss თითქმის ერთი გამოვიდა — მთავარი depth/reg იყო, არა loss.
- holiday flag-ების მოშორება XGBoost-ზე თითქმის არ იმოქმედა (base `IsHoliday` + წონა ჰყოფნის).